# **Modelos de Regressão - Introdução a Redes Neurais**

Neste primeiro dia, vamos começar com os fundamentos: **Regressão Linear** e **Gradiente Descendente**.
Esses conceitos são a base para entender como redes neurais aprendem.

## **1 - Importando bibliotecas necessárias**

In [ ]:
import sys
import numpy as np
import sklearn
import matplotlib.pyplot as plt
from packaging import version
from sklearn.linear_model import LinearRegression

Verifica versão do Python (mínimo 3.7)

In [ ]:
assert sys.version_info >= (3, 7)

Também requer o Scikit-Learn ≥ 1.0.1:

In [ ]:
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

Vamos definir os tamanhos de fonte padrão para deixar as figuras mais bonitas:

In [ ]:
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

Vamos criar a pasta `images/training_linear_models` (caso ainda não exista) e definir a função `save_fig()`, que será usada neste notebook para salvar as figuras em alta resolução para o livro:

In [ ]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "training_linear_models"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

## **2 - Linear Regression**

Como demosntramos, a regressão linear pode ser resolvida diretamente pela Equação Normal:

$$
\hat{\theta} = (X^{T}X)^{-1}X^{T}y
$$

Vamos criar uma base de dados artificial com 100 pontos seguindo uma relação linear com ruído.

In [ ]:
np.random.seed(42)  # Para resultados reproduzíveis
m = 100  # Número de exemplos
X = 2 * np.random.rand(m, 1)  # Variável independente
y = 4 + 3 * X + np.random.randn(m, 1)  # Variável dependente (y = 4 + 3x + ruído)

# np.random.randn gera amostras de uma distribuição normal (gaussiana)
# padrão (média 0, desvio padrão 1).

In [ ]:
# Plotando os dados gerados

plt.figure(figsize=(6, 4))
plt.plot(X, y, "b.")
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.axis([0, 2, 0, 15])
plt.grid()
save_fig("generated_data_plot")
plt.show()

A função `add_dummy_feature` da `sklearn.preprocessing` é usada para adicionar uma coluna de 1s (uns) a uma matriz de dados. Isso é comum e necessário em modelos de regressão linear (e outros modelos lineares) para representar o termo de intercepto (também conhecido como bias ou coeficiente de deslocamento).

In [ ]:
from sklearn.preprocessing import add_dummy_feature

# Adicionando coluna de 1s para o intercepto (bias)
X_b = add_dummy_feature(X)

# Calculando theta pelo método da equação normal
theta_best = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

In [ ]:
print("Parâmetros encontrados:")
print(f"Intercepto (theta0): {theta_best[0][0]:.4f}")
print(f"Inclinação (theta1): {theta_best[1][0]:.4f}")

In [ ]:
# Criando novos pontos para testar o modelo
X_new = np.array([[0], [2]])
X_new_b = add_dummy_feature(X_new)  # add x0 = 1 to each instance
y_predict = X_new_b @ theta_best
y_predict

In [ ]:
plt.figure(figsize=(6, 4))  # extra code – not needed, just formatting
plt.plot(X_new, y_predict, "r-", label="Predição")
plt.plot(X, y, "b.",  label="Dados reais")

# extra code – beautifies and saves Figure 4–2
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.axis([0, 2, 0, 15])
plt.grid()
plt.legend(loc="upper left")
save_fig("linear_model_predictions_plot")

plt.show()

Na prática, usamos bibliotecas prontas como `sklearn`, para mais informações entre no site deles: [Scikit-Learn](https://scikit-learn.org/stable/).

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)
print(f"Intercepto: {model.intercept_[0]:.4f}")
print(f"Coeficiente: {model.coef_[0][0]:.4f}")

In [ ]:
model.predict(X_new)

A classe `LinearRegression` é baseada na função `scipy.linalg.lstsq()` (o nome significa "mínimos quadrados"), que você pode chamar diretamente:

In [ ]:
theta_best_svd, residuals, rank, s = np.linalg.lstsq(X_b, y)
theta_best_svd

Esta função calcula $\mathbf{X}^+\mathbf{y}$, onde $\mathbf{X}^{+}$ é a _pseudoinversa_ de $\mathbf{X}$ (especificamente a inversa de Moore-Penrose). Você pode usar `np.linalg.pinv()` para calcular a pseudoinversa diretamente:

In [ ]:
np.linalg.pinv(X_b) @ y

## **3 - Gradiente Descendente**

O gradiente descendente é um algoritmo iterativo que busca minimizar a função de custo.
Ele é essencial em redes neurais, onde a solução analítica (equação normal) é inviável.

**Leitura Recomendada:** [Gradient Descent - Uma introdução visual](https://www.jeremyjordan.me/gradient-descent/)

### **3.1 - Descida de Gradiente em Lote**

In [ ]:
np.random.seed(42)
eta = 0.1                      # Taxa de aprendizado
n_epochs = 1000                # Número de épocas
theta = np.random.randn(2, 1)  # Inicialização aleatória
m = len(X_b)                   # number of instances

for epoch in range(n_epochs):
    gradients = (2 / m) * X_b.T @ (X_b @ theta - y)
    theta = theta - eta * gradients

Os parâmetros do modelo treinado:

In [ ]:
print("Parâmetros finais (Batch GD):")
print(theta)

In [ ]:
# Código extra – gera e salva a Figura 4–8

import matplotlib as mpl

def plot_gradient_descent(theta, eta):
    # Número de instâncias
    m = len(X_b)
    # Plota os dados originais
    plt.plot(X, y, "b.")
    # Número total de épocas (para o loop)
    n_epochs = 1000
    # Número de passos do gradiente descendente a serem mostrados no gráfico
    n_shown = 20
    # Lista para armazenar o caminho de theta no espaço de parâmetros
    theta_path = []
    for epoch in range(n_epochs):
        # Se a época for uma das primeiras a serem mostradas
        if epoch < n_shown:
            # Calcula as previsões com o theta atual para os novos pontos
            y_predict = X_new_b @ theta
            # Gera uma cor para a linha de previsão
            color = mpl.colors.rgb2hex(plt.cm.OrRd(epoch / n_shown + 0.15))
            # Plota a linha de previsão
            plt.plot(X_new, y_predict, linestyle="solid", color=color)
        # Calcula os gradientes da função de custo
        gradients = 2 / m * X_b.T @ (X_b @ theta - y)
        # Atualiza os parâmetros theta usando o gradiente descendente
        theta = theta - eta * gradients
        # Adiciona o theta atual ao caminho
        theta_path.append(theta)
    # Rótulo do eixo X
    plt.xlabel("$x_1$")
    # Define os limites dos eixos
    plt.axis([0, 2, 0, 15])
    # Adiciona grade ao gráfico
    plt.grid()
    # Adiciona título ao gráfico mostrando a taxa de aprendizado (eta)
    plt.title(fr"$\eta = {eta}$")
    # Retorna o caminho de theta
    return theta_path

# Define a semente para reprodutibilidade
np.random.seed(42)
# Inicializa theta aleatoriamente
theta = np.random.randn(2, 1)  # inicialização aleatória

# Cria uma figura com tamanho específico
plt.figure(figsize=(10, 4))
# Cria o primeiro subplot (1 linha, 3 colunas, 1ª posição)
plt.subplot(131)
# Chama a função para plotar o gradiente descendente com eta=0.02
plot_gradient_descent(theta, eta=0.02)
# Rótulo do eixo Y
plt.ylabel("$y$", rotation=0)
# Cria o segundo subplot (1 linha, 3 colunas, 2ª posição)
plt.subplot(132)
# Chama a função para plotar o gradiente descendente com eta=0.1 e armazena o caminho
theta_path_bgd = plot_gradient_descent(theta, eta=0.1)
# Remove os rótulos do eixo Y para este subplot
plt.gca().axes.yaxis.set_ticklabels([])
# Cria o terceiro subplot (1 linha, 3 colunas, 3ª posição)
plt.subplot(133)
# Remove os rótulos do eixo Y para este subplot
plt.gca().axes.yaxis.set_ticklabels([])
# Chama a função para plotar o gradiente descendente com eta=0.5
plot_gradient_descent(theta, eta=0.5)
# Salva a figura
save_fig("gradient_descent_plot")
# Exibe a figura
plt.show()

### **3.2 - Descida de Gradiente Estocástica (SGD)**

O **Stochastic Gradient Descent** (SGD) ou **Gradiente Descendente Estocástico** é uma variação do algoritmo de gradiente descendente usado para otimizar funções de custo em machine learning. Diferente da versão em lote, o SGD atualiza os parâmetros do modelo usando **apenas um exemplo de treinamento por vez**, em vez de todo o conjunto de dados.

Para regressão linear, usamos o Erro Quadrático Médio:

$$MSE(\theta) = \frac{1}{m} \sum_{i=1}^{m} (\theta^T \cdot x^{(i)} - y^{(i)})^2$$

Onde:
- $m$ = número de exemplos
- $\theta$ = vetor de parâmetros
- $x^{(i)}$ = características do i-ésimo exemplo
- $y^{(i)}$ = valor real do i-ésimo exemplo

O gradiente do MSE em relação aos parâmetros é:

$$\nabla_{\theta} MSE(\theta) = \frac{2}{m} \sum_{i=1}^{m} (\theta^T \cdot x^{(i)} - y^{(i)}) \cdot x^{(i)}$$

In [ ]:
# ============================================
# 1. CONFIGURAÇÕES INICIAIS
# ============================================

from sklearn.metrics import mean_squared_error

# Gerando dados sintéticos (os mesmos do notebook)
np.random.seed(42)
m = 100  # Número de instâncias
X = 2 * np.random.rand(m, 1)  # Variável independente
y = 4 + 3 * X + np.random.randn(m, 1)  # y = 4 + 3x + ruído

# Adicionando coluna de 1s para o intercepto (bias)
X_b = np.c_[np.ones((m, 1)), X]

# Parâmetros do SGD
n_epochs = 50
t0, t1 = 5, 50  # Hiperparâmetros do schedule de aprendizado

def learning_schedule(t):
    """Função de agendamento da taxa de aprendizado (decai com o tempo)"""
    return t0 / (t + t1)

# Inicialização dos parâmetros
np.random.seed(42)
theta = np.random.randn(2, 1)  # [intercepto, inclinação]

In [ ]:
# ============================================
# 2. ESTRUTURAS DE DADOS PARA ARMAZENAR HISTÓRICO
# ============================================

# Listas para armazenar o histórico
historico_thetas = []      # Todos os parâmetros (theta0, theta1)
historico_perdas = []      # Perda (MSE) após cada atualização
historico_etas = []        # Taxa de aprendizado usada em cada iteração
historico_iteracoes = []   # Número da iteração global

# Para calcular a perda (MSE) a cada iteração
def calcular_mse(theta, X_b, y):
    """Calcula o Erro Quadrático Médio (MSE)"""
    y_pred = X_b @ theta
    return mean_squared_error(y, y_pred)

In [ ]:
# ============================================
# 3. TREINAMENTO COM REGISTRO DE HISTÓRICO
# ============================================
np.random.seed(42)

print("=" * 70)
print("INICIANDO TREINAMENTO DO SGD")
print("=" * 70)
print(f"{'Iteração':<10} {'Época':<8} {'Theta0':<12} {'Theta1':<12} {'Eta':<10} {'MSE':<12}")
print("-" * 70)

iteracao_global = 0

for epoch in range(n_epochs):
    for iteration in range(m):

        # 1. Amostragem estocástica (escolhe 1 exemplo aleatório)
        random_index = np.random.randint(m)
        xi = X_b[random_index:random_index + 1]
        yi = y[random_index:random_index + 1]

        # 2. Calcula gradiente para este único exemplo
        gradientes = 2 * xi.T @ (xi @ theta - yi)

        # 3. Calcula taxa de aprendizado atual
        eta = learning_schedule(iteracao_global)

        # 4. Atualiza os parâmetros
        theta_antes = theta.copy()
        theta = theta - eta * gradientes

        # 5. Calcula a perda (MSE) após a atualização
        perda_atual = calcular_mse(theta, X_b, y)

        # 6. ARMAZENA HISTÓRICO
        historico_thetas.append(theta.copy())
        historico_perdas.append(perda_atual)
        historico_etas.append(eta)
        historico_iteracoes.append(iteracao_global)

        # 7. Exibe progresso (a cada 100 iterações ou nas primeiras 10)
        if iteracao_global < 10 or iteracao_global % 100 == 0:
            print(f"{iteracao_global:<10} {epoch:<8} "
                  f"{theta[0,0]:<12.4f} {theta[1,0]:<12.4f} "
                  f"{eta:<10.6f} {perda_atual:<12.6f}")

        iteracao_global += 1

print("-" * 70)
print(f"TREINAMENTO CONCLUÍDO! Total de iterações: {iteracao_global}")
print("=" * 70)

Se o dataset tem m = 100 exemplos:

1 época = 100 iterações

1 iteração = 1 exemplo (tamanho do batch = 1)

Total de atualizações = 50 × 100 = 5.000 atualizações

In [ ]:
# ============================================
# 4. VISUALIZAÇÕES COMPLETAS
# ============================================

# Configurando estilo dos gráficos
plt.style.use('seaborn-v0_8-darkgrid')
fig = plt.figure(figsize=(15, 12))

# 4.1 GRÁFICO 1: Evolução dos Parâmetros (Theta0 e Theta1)
ax1 = plt.subplot(3, 2, 1)
ax1.plot(historico_iteracoes, [t[0,0] for t in historico_thetas],
         'b-', linewidth=2, label='Theta0 (Intercepto)')
ax1.plot(historico_iteracoes, [t[1,0] for t in historico_thetas],
         'r-', linewidth=2, label='Theta1 (Inclinação)')
ax1.set_xlabel('Iteração')
ax1.set_ylabel('Valor do Parâmetro')
ax1.set_title('Evolução dos Parâmetros do Modelo')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 4.2 GRÁFICO 2: Evolução da Função Perda (MSE)
ax2 = plt.subplot(3, 2, 2)
ax2.semilogy(historico_iteracoes, historico_perdas, 'g-', linewidth=2)
ax2.set_xlabel('Iteração')
ax2.set_ylabel('MSE (escala log)')
ax2.set_title('Evolução da Função Perda (MSE)')
ax2.grid(True, alpha=0.3)

# 4.3 GRÁFICO 3: Taxa de Aprendizado ao Longo do Tempo
ax3 = plt.subplot(3, 2, 3)
ax3.plot(historico_iteracoes, historico_etas, 'purple', linewidth=2)
ax3.set_xlabel('Iteração')
ax3.set_ylabel('Taxa de Aprendizado (η)')
ax3.set_title('Decaimento da Taxa de Aprendizado')
ax3.grid(True, alpha=0.3)

# 4.4 GRÁFICO 4: Trajetória dos Parâmetros no Espaço 2D
ax4 = plt.subplot(3, 2, 4)
theta0_vals = [t[0,0] for t in historico_thetas]
theta1_vals = [t[1,0] for t in historico_thetas]
ax4.plot(theta0_vals, theta1_vals, 'b-', linewidth=1, alpha=0.7)
ax4.scatter(theta0_vals[0], theta1_vals[0], color='green', s=100,
            label='Início', zorder=5)
ax4.scatter(theta0_vals[-1], theta1_vals[-1], color='red', s=100,
            label='Final', zorder=5)
ax4.set_xlabel('Theta0 (Intercepto)')
ax4.set_ylabel('Theta1 (Inclinação)')
ax4.set_title('Trajetória dos Parâmetros no Espaço 2D')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 4.5 GRÁFICO 5: Comparação com a Solução Ótima (Equação Normal)
ax5 = plt.subplot(3, 2, 5)
# Calcula a solução ótima pela equação normal
theta_otimo = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

# Cria pontos para a reta de regressão
X_plot = np.array([[0], [2]])
X_plot_b = np.c_[np.ones((2, 1)), X_plot]

# Reta inicial (primeira iteração)
y_inicial = X_plot_b @ historico_thetas[0]

# Reta final (última iteração)
y_final = X_plot_b @ historico_thetas[-1]

# Reta ótima (equação normal)
y_otimo = X_plot_b @ theta_otimo

# Plot dos dados
ax5.scatter(X, y, alpha=0.5, label='Dados reais')

# Plot das retas
ax5.plot(X_plot, y_inicial, 'g--', linewidth=2, label='Modelo inicial')
ax5.plot(X_plot, y_final, 'b-', linewidth=2, label='Modelo final (SGD)')
ax5.plot(X_plot, y_otimo, 'r:', linewidth=2, label='Solução ótima')

ax5.set_xlabel('x')
ax5.set_ylabel('y')
ax5.set_title('Comparação: Modelo Inicial vs. Final vs. Ótimo')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 4.6 GRÁFICO 6: Zoom nas primeiras iterações (para visualizar a convergência inicial)
ax6 = plt.subplot(3, 2, 6)
n_zoom = min(200, len(historico_iteracoes))
ax6.plot(historico_iteracoes[:n_zoom],
         [t[0,0] for t in historico_thetas[:n_zoom]],
         'b-', label='Theta0', linewidth=2)
ax6.plot(historico_iteracoes[:n_zoom],
         [t[1,0] for t in historico_thetas[:n_zoom]],
         'r-', label='Theta1', linewidth=2)
ax6.set_xlabel('Iteração')
ax6.set_ylabel('Valor do Parâmetro')
ax6.set_title(f'Primeiras {n_zoom} Iterações (Zoom na Convergência)')
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# 5. TABELA RESUMO DOS PRINCIPAIS MARCOS
# ============================================

print("\n" + "=" * 70)
print("RESUMO DOS PRINCIPAIS MARCOS DO TREINAMENTO")
print("=" * 70)

# Índices importantes
indices_marcos = [0, 10, 50, 100, 500, 1000, -1]
marcos = ["Início", "Após 10 iterações", "Após 50 iterações",
          "Após 100 iterações", "Após 500 iterações", "Após 1000 iterações", "Final"]

print(f"\n{'Marco':<25} {'Theta0':<12} {'Theta1':<12} {'MSE':<12} {'Eta':<12}")
print("-" * 70)

for idx, marco_idx in enumerate(indices_marcos):
    if marco_idx == -1:
        marco_idx = len(historico_thetas) - 1

    theta_at = historico_thetas[marco_idx]
    perda_at = historico_perdas[marco_idx]
    eta_at = historico_etas[marco_idx]

    print(f"{marcos[idx]:<25} "
          f"{theta_at[0,0]:<12.6f} "
          f"{theta_at[1,0]:<12.6f} "
          f"{perda_at:<12.6f} "
          f"{eta_at:<12.6f}")

print("=" * 70)

In [ ]:
# ============================================
# 6. ESTATÍSTICAS FINAIS
# ============================================

print("\n" + "=" * 70)
print("ESTATÍSTICAS FINAIS DO MODELO")
print("=" * 70)

theta_final = historico_thetas[-1]
theta_otimo = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

print(f"\n📊 PARÂMETROS FINAIS (SGD):")
print(f"   Intercepto (Theta0): {theta_final[0,0]:.6f}")
print(f"   Inclinação (Theta1): {theta_final[1,0]:.6f}")

print(f"\n📊 PARÂMETROS ÓTIMOS (Equação Normal):")
print(f"   Intercepto (Theta0): {theta_otimo[0,0]:.6f}")
print(f"   Inclinação (Theta1): {theta_otimo[1,0]:.6f}")

print(f"\n📊 DIFERENÇA ENTRE SGD E ÓTIMO:")
print(f"   ΔTheta0: {abs(theta_final[0,0] - theta_otimo[0,0]):.6f}")
print(f"   ΔTheta1: {abs(theta_final[1,0] - theta_otimo[1,0]):.6f}")

print(f"\n📊 MÉTRICAS DE TREINAMENTO:")
print(f"   Total de iterações: {len(historico_iteracoes)}")
print(f"   MSE inicial: {historico_perdas[0]:.6f}")
print(f"   MSE final: {historico_perdas[-1]:.6f}")
print(f"   Redução do MSE: {(1 - historico_perdas[-1]/historico_perdas[0])*100:.2f}%")
print(f"   Eta inicial: {historico_etas[0]:.6f}")
print(f"   Eta final: {historico_etas[-1]:.6f}")

print("=" * 70)

Embora seja fundamental entender a matemática e implementar o SGD manualmente para aprendizado, em projetos reais utilizamos bibliotecas otimizadas como o Scikit-Learn. Elas oferecem:

- ✅ **Código testado e validado** pela comunidade
- ✅ **Otimizações de performance** (Cython, paralelismo)
- ✅ **Muitos recursos extras** (regularização, early stopping, etc.)
- ✅ **Menos bugs e edge cases tratados**
- ✅ **Integração com outras ferramentas** (pipelines, validação cruzada)

In [ ]:
from sklearn.linear_model import SGDRegressor

# Cria o modelo com hiperparâmetros configurados
sgd_reg = SGDRegressor(
    max_iter=1000,        # Número máximo de épocas
    tol=1e-5,             # Tolerância para parada antecipada
    penalty=None,         # Tipo de regularização (None, 'l2', 'l1', 'elasticnet')
    eta0=0.01,            # Taxa de aprendizado inicial
    n_iter_no_change=100, # Épocas sem melhora para early stopping
    random_state=42       # Semente para reprodutibilidade
)

# Treina o modelo (y.ravel() transforma para 1D)
sgd_reg.fit(X, y.ravel())

In [ ]:
sgd_reg.intercept_, sgd_reg.coef_

### **3.3 - Descida de gradiente em mini-lotes (MGD)**

O **Mini-Batch Gradient Descent** (MGD) ou **Gradiente Descendente em Mini-Lotes** é um algoritmo de otimização que representa um **meio-termo** entre o Batch GD (usando todos os dados) e o SGD (usando 1 exemplo). Ele atualiza os parâmetros usando **um pequeno lote (batch) de exemplos** a cada iteração.

In [ ]:
# ============================================
# 1. PREPARAÇÃO DOS DADOS
# ============================================

from math import ceil

# Gerando dados sintéticos
np.random.seed(42)
m = 100  # Número de instâncias
X = 2 * np.random.rand(m, 1)
y = 4 + 3 * X + np.random.randn(m, 1)

# Adicionando coluna de 1s para o intercepto
X_b = np.c_[np.ones((m, 1)), X]

# ============================================
# 2. CONFIGURAÇÕES DO MINI-BATCH
# ============================================

n_epochs = 50
minibatch_size = 20  # Tamanho do lote (20 exemplos)
n_batches_per_epoch = ceil(m / minibatch_size)  # 5 batches/época

print(f"Dataset: {m} exemplos")
print(f"Batch size: {minibatch_size}")
print(f"Batches por época: {n_batches_per_epoch}")
print(f"Total de atualizações: {n_epochs * n_batches_per_epoch}")

In [ ]:
# ============================================
# 3. INICIALIZAÇÃO
# ============================================

np.random.seed(42)
theta = np.random.randn(2, 1)  # [intercepto, inclinação]

# Schedule de aprendizado (decai mais lentamente que no SGD)
t0, t1 = 200, 1000

def learning_schedule(t):
    """Taxa de aprendizado decrescente para o MGD"""
    return t0 / (t + t1)

# ============================================
# 4. ESTRUTURAS PARA HISTÓRICO
# ============================================

historico_theta0 = []      # Armazena apenas o valor do intercepto (escalar)
historico_theta1 = []      # Armazena apenas o valor da inclinação (escalar)
historico_perdas = []
historico_etas = []
historico_iteracoes = []

def calcular_mse(theta, X_b, y):
    """Calcula o Erro Quadrático Médio"""
    y_pred = X_b @ theta
    return mean_squared_error(y, y_pred)


In [ ]:
# ============================================
# 5. TREINAMENTO MINI-BATCH
# ============================================

print("\n" + "=" * 80)
print("TREINAMENTO MINI-BATCH GRADIENT DESCENT")
print("=" * 80)
print(f"{'Iteração':<10} {'Época':<8} {'Batch':<8} {'Theta0':<12} {'Theta1':<12} {'Eta':<10} {'MSE':<12}")
print("-" * 80)

iteracao_global = 0
theta_path_mgd = []  # Para compatibilidade com código original

for epoch in range(n_epochs):
    # 🔀 Embaralha os dados no início de cada época
    shuffled_indices = np.random.permutation(m)
    X_b_shuffled = X_b[shuffled_indices]
    y_shuffled = y[shuffled_indices]

    # 📦 Processa cada batch da época
    for batch_idx in range(n_batches_per_epoch):
        # Seleciona os índices do batch atual
        inicio = batch_idx * minibatch_size
        fim = min(inicio + minibatch_size, m)
        xi = X_b_shuffled[inicio:fim]
        yi = y_shuffled[inicio:fim]

        # Tamanho real do batch (pode ser menor no último batch)
        batch_real = len(xi)

        # 📐 Calcula o gradiente para o mini-batch
        gradients = (2 / batch_real) * xi.T @ (xi @ theta - yi)

        # 📉 Calcula taxa de aprendizado (decaimento suave)
        eta = learning_schedule(iteracao_global)

        # 🔄 Atualiza os parâmetros
        theta = theta - eta * gradients

        # 📊 Calcula perda atual (MSE)
        perda_atual = calcular_mse(theta, X_b, y)

         # Armazena histórico (VALORES ESCALARES!)
        historico_theta0.append(float(theta[0, 0]))  # Converte para float escalar
        historico_theta1.append(float(theta[1, 0]))  # Converte para float escalar
        historico_perdas.append(float(perda_atual))
        historico_etas.append(float(eta))
        historico_iteracoes.append(iteracao_global)
        theta_path_mgd.append([float(theta[0, 0]), float(theta[1, 0])])  # Lista com 2 floats

        # 📝 Exibe progresso
        if iteracao_global < 10 or iteracao_global % 50 == 0:
            print(f"{iteracao_global:<10} {epoch:<8} {batch_idx:<8} "
                  f"{theta[0,0]:<12.4f} {theta[1,0]:<12.4f} "
                  f"{eta:<10.6f} {perda_atual:<12.6f}")

        iteracao_global += 1

print("-" * 80)
print(f"TREINAMENTO CONCLUÍDO! Total de atualizações: {iteracao_global}")
print("=" * 80)

# ============================================
# CONVERTENDO PARA ARRAYS NUMPY PARA FACILITAR INDEXAÇÃO
# ============================================

In [ ]:
# Convertendo listas para arrays numpy
# Convertendo listas para arrays numpy (já são valores escalares)
historico_theta0 = np.array(historico_theta0)
historico_theta1 = np.array(historico_theta1)
historico_perdas = np.array(historico_perdas)
historico_etas = np.array(historico_etas)
historico_iteracoes = np.array(historico_iteracoes)
theta_path_mgd = np.array(theta_path_mgd)  # Shape: (n_iteracoes, 2)

print(f"\n🔍 Shapes dos arrays:")
print(f"   historico_theta0: {historico_theta0.shape}")
print(f"   historico_theta1: {historico_theta1.shape}")
print(f"   historico_perdas: {historico_perdas.shape}")
print(f"   theta_path_mgd: {theta_path_mgd.shape}")

In [ ]:
# ============================================
# 6. SIMULAÇÃO DOS OUTROS MÉTODOS PARA COMPARAÇÃO
# ============================================

print("\n🔧 Simulando SGD e Batch GD para comparação...")

# Simulando SGD (para fins de comparação)
np.random.seed(42)
theta_sgd = np.random.randn(2, 1)
theta_path_sgd = []

for epoch in range(n_epochs):
    for i in range(m):
        idx = np.random.randint(m)
        xi = X_b[idx:idx+1]
        yi = y[idx:idx+1]
        grad = 2 * xi.T @ (xi @ theta_sgd - yi)
        eta = 5 / (epoch * m + i + 50)
        theta_sgd -= eta * grad
        if i % 20 == 0:  # Amostragem para não ficar muito grande
            theta_path_sgd.append(theta_sgd.copy().flatten())

theta_path_sgd = np.array(theta_path_sgd)

# Simulando Batch GD
np.random.seed(42)
theta_bgd = np.random.randn(2, 1)
theta_path_bgd = []
eta_bgd = 0.1

for epoch in range(50):
    grad = (2/m) * X_b.T @ (X_b @ theta_bgd - y)
    theta_bgd -= eta_bgd * grad
    theta_path_bgd.append(theta_bgd.copy().flatten())

theta_path_bgd = np.array(theta_path_bgd)

print(f"   theta_path_sgd shape: {theta_path_sgd.shape}")
print(f"   theta_path_bgd shape: {theta_path_bgd.shape}")
print("✅ Simulação concluída!")

In [ ]:
# ============================================
# 7. VISUALIZAÇÕES DETALHADAS
# ============================================

plt.style.use('seaborn-v0_8-darkgrid')
fig = plt.figure(figsize=(16, 12))

# 7.1 Evolução dos Parâmetros
ax1 = plt.subplot(3, 3, 1)
ax1.plot(historico_iteracoes, historico_theta0, 'b-', linewidth=2, label='Theta0 (Intercepto)')
ax1.plot(historico_iteracoes, historico_theta1, 'r-', linewidth=2, label='Theta1 (Inclinação)')
ax1.set_xlabel('Iteração (atualização do batch)')
ax1.set_ylabel('Valor do Parâmetro')
ax1.set_title('Evolução dos Parâmetros (MGD)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 7.2 Evolução da Perda (MSE)
ax2 = plt.subplot(3, 3, 2)
ax2.semilogy(historico_iteracoes, historico_perdas, 'g-', linewidth=2)
ax2.set_xlabel('Iteração')
ax2.set_ylabel('MSE (escala log)')
ax2.set_title('Evolução do Erro (MSE)')
ax2.grid(True, alpha=0.3)

# 7.3 Taxa de Aprendizado
ax3 = plt.subplot(3, 3, 3)
ax3.plot(historico_iteracoes, historico_etas, 'purple', linewidth=2)
ax3.set_xlabel('Iteração')
ax3.set_ylabel('Taxa de Aprendizado (η)')
ax3.set_title('Decaimento da Taxa de Aprendizado')
ax3.grid(True, alpha=0.3)

# 7.4 Trajetória no Espaço 2D (Mini-Batch)
ax4 = plt.subplot(3, 3, 4)
ax4.plot(historico_theta0, historico_theta1, 'g-+',
         linewidth=1.5, markersize=3, alpha=0.7, label='Trajetória MGD')
ax4.scatter(historico_theta0[0], historico_theta1[0],
            color='green', s=100, label='Início', zorder=5)
ax4.scatter(historico_theta0[-1], historico_theta1[-1],
            color='red', s=100, label='Final', zorder=5)
ax4.set_xlabel('Theta0 (Intercepto)')
ax4.set_ylabel('Theta1 (Inclinação)')
ax4.set_title('Trajetória dos Parâmetros (MGD)')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 7.5 Estabilidade da Convergência
ax5 = plt.subplot(3, 3, 5)
# Calcula a variância do gradiente (magnitude das mudanças)
if len(historico_theta0) > 1:
    grad_variance = np.abs(np.diff(historico_theta0))
    ax5.plot(grad_variance, 'b-', alpha=0.7)
    ax5.axhline(y=np.mean(grad_variance), color='r', linestyle='--',
                label=f'Média: {np.mean(grad_variance):.4f}')
ax5.set_xlabel('Iteração')
ax5.set_ylabel('Magnitude do passo')
ax5.set_title('Estabilidade da Convergência')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 7.6 Histograma dos parâmetros finais
ax6 = plt.subplot(3, 3, 6)
n_last = min(50, len(historico_theta0))
ax6.hist(historico_theta0[-n_last:], bins=15, alpha=0.7, label='Theta0 final', color='blue')
ax6.hist(historico_theta1[-n_last:], bins=15, alpha=0.7, label='Theta1 final', color='red')
ax6.set_xlabel('Valor do Parâmetro')
ax6.set_ylabel('Frequência')
ax6.set_title(f'Distribuição dos Parâmetros (últimas {n_last} iterações)')
ax6.legend()
ax6.grid(True, alpha=0.3)

# 7.7 COMPARAÇÃO DOS TRÊS MÉTODOS
ax7 = plt.subplot(3, 3, 7)

# Plot SGD (se existir)
if len(theta_path_sgd) > 0:
    ax7.plot(theta_path_sgd[:, 0], theta_path_sgd[:, 1], 'r-s',
             linewidth=1, markersize=3, label='SGD', alpha=0.7)

# Plot Batch GD (se existir)
if len(theta_path_bgd) > 0:
    ax7.plot(theta_path_bgd[:, 0], theta_path_bgd[:, 1], 'b-o',
             linewidth=1.5, markersize=4, label='Batch GD', alpha=0.7)

# Plot Mini-Batch
ax7.plot(historico_theta0, historico_theta1, 'g-+',
         linewidth=1.5, markersize=3, label='Mini-Batch (MGD)', alpha=0.7)

ax7.set_xlabel('Theta0 (Intercepto)')
ax7.set_ylabel('Theta1 (Inclinação)')
ax7.set_title('Comparação: MGD vs SGD vs Batch GD')
ax7.legend()
ax7.grid(True, alpha=0.3)

# 7.8 Zoom nas primeiras iterações
ax8 = plt.subplot(3, 3, 8)
n_zoom = min(50, len(historico_iteracoes))
ax8.plot(historico_iteracoes[:n_zoom], historico_theta0[:n_zoom],
         'b-', linewidth=2, label='Theta0')
ax8.plot(historico_iteracoes[:n_zoom], historico_theta1[:n_zoom],
         'r-', linewidth=2, label='Theta1')
ax8.set_xlabel('Iteração')
ax8.set_ylabel('Valor do Parâmetro')
ax8.set_title(f'Primeiras {n_zoom} Iterações (Zoom)')
ax8.legend()
ax8.grid(True, alpha=0.3)

# 7.9 Reta de Regressão Final
ax9 = plt.subplot(3, 3, 9)
# Dados originais
ax9.scatter(X, y, alpha=0.5, label='Dados reais')
# Reta final do modelo
X_plot = np.array([[0], [2]])
X_plot_b = np.c_[np.ones((2, 1)), X_plot]
# Usando theta_final corretamente (como array 2D para multiplicação)
theta_final_2d = np.array([[historico_theta0[-1]], [historico_theta1[-1]]])
y_final = X_plot_b @ theta_final_2d
ax9.plot(X_plot, y_final, 'g-', linewidth=3, label=f'MGD final')
# Solução ótima (equação normal)
theta_otimo = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
y_otimo = X_plot_b @ theta_otimo
ax9.plot(X_plot, y_otimo, 'r--', linewidth=2, label='Solução ótima')
ax9.set_xlabel('x')
ax9.set_ylabel('y')
ax9.set_title('Modelo Final vs. Solução Ótima')
ax9.legend()
ax9.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# 8. ESTATÍSTICAS FINAIS
# ============================================

print("\n" + "=" * 80)
print("ESTATÍSTICAS DO MINI-BATCH GRADIENT DESCENT")
print("=" * 80)

# Usando os valores escalares diretamente
theta0_final = historico_theta0[-1]  # Já é um float escalar
theta1_final = historico_theta1[-1]  # Já é um float escalar

theta_otimo = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

print(f"\n📊 PARÂMETROS FINAIS (MGD):")
print(f"   Intercepto (Theta0): {theta0_final:.6f}")
print(f"   Inclinação (Theta1): {theta1_final:.6f}")

print(f"\n📊 PARÂMETROS ÓTIMOS (Equação Normal):")
print(f"   Intercepto (Theta0): {theta_otimo[0,0]:.6f}")
print(f"   Inclinação (Theta1): {theta_otimo[1,0]:.6f}")

print(f"\n📊 DIFERENÇA ENTRE MGD E ÓTIMO:")
print(f"   ΔTheta0: {abs(theta0_final - theta_otimo[0,0]):.6f}")
print(f"   ΔTheta1: {abs(theta1_final - theta_otimo[1,0]):.6f}")

print(f"\n📊 MÉTRICAS DE TREINAMENTO:")
print(f"   Total de atualizações: {len(historico_iteracoes)}")
print(f"   MSE inicial: {historico_perdas[0]:.6f}")
print(f"   MSE final: {historico_perdas[-1]:.6f}")
print(f"   Redução do MSE: {(1 - historico_perdas[-1]/historico_perdas[0])*100:.2f}%")
print(f"   Eta inicial: {historico_etas[0]:.6f}")
print(f"   Eta final: {historico_etas[-1]:.6f}")

print(f"\n📊 EFICIÊNCIA COMPUTACIONAL:")
print(f"   Atualizações por época: {n_batches_per_epoch}")
print(f"   Total de épocas: {n_epochs}")
print(f"   Total de gradientes calculados: {n_epochs * n_batches_per_epoch}")
print(f"   Equivalente em SGD: {m * n_epochs} atualizações")
print(f"   Economia: {(1 - (n_batches_per_epoch/m)) * 100:.1f}% menos atualizações que SGD")

print(f"\n📊 COMPARAÇÃO COM OUTROS MÉTODOS:")
print(f"   Batch GD: 1 atualização por época (mais estável, mais lento)")
print(f"   SGD: {m} atualizações por época (mais rápido, mais ruidoso)")
print(f"   MGD: {n_batches_per_epoch} atualizações por época (equilibrado)")

In [ ]:
# ============================================
# 9. TABELA DE MARCOS
# ============================================

print("\n" + "=" * 80)
print("MARCOS DO TREINAMENTO")
print("=" * 80)

# Garantindo que os índices são válidos
total_iteracoes = len(historico_theta0)
indices_marcos = [0, min(10, total_iteracoes-1),
                  min(50, total_iteracoes-1), min(100, total_iteracoes-1),
                  total_iteracoes//2, total_iteracoes-1]
marcos = ["Início", "10 iterações", "50 iterações",
          "100 iterações", "Metade do treino", "Final"]

print(f"\n{'Marco':<20} {'Theta0':<12} {'Theta1':<12} {'MSE':<12} {'Eta':<12}")
print("-" * 70)

for idx, marco_idx in enumerate(indices_marcos):
    if marco_idx >= total_iteracoes:
        marco_idx = total_iteracoes - 1
    print(f"{marcos[idx]:<20} "
          f"{historico_theta0[marco_idx]:<12.6f} "
          f"{historico_theta1[marco_idx]:<12.6f} "
          f"{historico_perdas[marco_idx]:<12.6f} "
          f"{historico_etas[marco_idx]:<12.6f}")

print("=" * 80)


In [ ]:
# ============================================
# 10. PLOT COMPARATIVO FINAL
# ============================================

plt.figure(figsize=(10, 6))

# Verificando se os arrays existem e têm dados
if len(theta_path_sgd) > 0:
    plt.plot(theta_path_sgd[:, 0], theta_path_sgd[:, 1], 'r-s', linewidth=1,
             markersize=3, label='Stochastic (SGD)', alpha=0.7)

if len(theta_path_mgd) > 0:
    plt.plot(theta_path_mgd[:, 0], theta_path_mgd[:, 1], 'g-+', linewidth=1.5,
             markersize=3, label='Mini-batch (MGD)', alpha=0.7)

if len(theta_path_bgd) > 0:
    plt.plot(theta_path_bgd[:, 0], theta_path_bgd[:, 1], 'b-o', linewidth=2,
             markersize=4, label='Batch (BGD)', alpha=0.7)

plt.xlabel(r'$\theta_0$ (Intercepto)')
plt.ylabel(r'$\theta_1$ (Inclinação)', rotation=90)
plt.title('Comparação das Trajetórias: SGD vs MGD vs Batch GD')
plt.legend(loc='upper left')
plt.axis([2.6, 4.6, 2.3, 3.4])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Análise completa do Mini-Batch Gradient Descent finalizada!")